# 03 — Final Evaluation

**Purpose.** Compare every model on the held-out test set, once, under
identical conditions.

**Inputs.** `data/processed/test.parquet` and the artifacts in `models/`.

**Outputs.** `reports/results/` and `reports/figures/`.

---

### Test-set discipline

This notebook is the **first and only** time the test split is read since
notebook 01 created it.

- Every model is scored on the same records, with the same metric function.
- Artifacts are loaded and used as-is: `transform`, never `fit`.
- If a result disappoints, the fix belongs in a `02x` notebook — and the test
  estimate is then compromised for every choice it informed. Tuning against the
  test set turns it into a second validation set, silently.

Run this once, when all models are final.

## 1. Setup

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from src.config import load_config
from src.utils.seed import set_seed

cfg = load_config()
set_seed(cfg.seed)

pd.set_option("display.max_columns", 50)
cfg

## 2. Load the test split

The frozen one, written by notebook 01.

In [ ]:
from src.data.load import load_processed

test_df = load_processed(cfg, "test")
target = cfg.target
X_test, y_test = test_df.drop(columns=[target]), test_df[target]
print(f"{len(test_df):,} test records")

## 3. Load every model artifact

Each artifact carries its own fitted preprocessor, so the models can differ
completely inside while being called identically here.

In [ ]:
from src.utils.io import artifact_metadata, load_artifact

MODELS = ["model_a"]  # TODO: list every model to compare

artifacts = {name: load_artifact(f"models/{name}.pkl") for name in MODELS}
{name: artifact_metadata(f"models/{name}.pkl") for name in MODELS}

## 4. Predict — the same test set, the same call

The uniform loop is possible because every model implements the same interface.

In [ ]:
predictions = {}
for name, artifact in artifacts.items():
    X = artifact["preprocessor"].transform(X_test)   # transform, never fit
    predictions[name] = artifact["model"].predict(X)

{name: preds[:5] for name, preds in predictions.items()}

## 5. Results table

One metric function, applied identically to every model.

In [ ]:
from src.evaluation.metrics import PRIMARY_METRIC, evaluate

results = pd.DataFrame(
    {name: evaluate(y_test.to_numpy(), preds) for name, preds in predictions.items()}
).T
results.sort_values(PRIMARY_METRIC)

## 6. Comparison plots

Show the spread, not only the point estimate. A bar chart of single numbers
invites over-reading differences that sit inside the noise.

In [ ]:
from src.evaluation.analysis import comparison_plot

comparison_plot(results, PRIMARY_METRIC)

# TODO: bootstrap the test set to put an interval around each score, so the
#       ranking can be read honestly.

## 7. Error analysis

Where each model fails, and whether they fail on the same records. Models that
fail differently are candidates for an ensemble; models that fail identically
share a data problem.

In [ ]:
from src.evaluation.analysis import calibration_plot, error_table, residual_plot

for name, preds in predictions.items():
    print(f"--- {name} ---")
    display(error_table(y_test.to_numpy(), preds, features=X_test, top_n=10))

In [ ]:
# TODO: residual_plot (regression) or calibration_plot (classification) per model
# TODO: overlap of the worst-predicted records across models

## 8. Per-segment breakdown

An aggregate score can hide a model that is excellent on the common case and
unusable on the segment that matters.

In [ ]:
from src.evaluation.metrics import evaluate_by_group

# TODO: choose the segmentation that matters operationally — site, device,
#       class band, time window — and score each model within it.

## 9. Export

Everything a report or a decision meeting needs, written to disk rather than
left in notebook output.

In [ ]:
from pathlib import Path

Path("reports/results").mkdir(parents=True, exist_ok=True)
results.to_csv("reports/results/model_comparison.csv")

# TODO: save each figure to reports/figures/ at publication resolution
print("wrote reports/results/model_comparison.csv")

## 10. Conclusions

State the decision and the reasoning, not just the winning row.

- **Selected model:** `<name>`
- **Why:** `<accuracy, and the cost / latency / interpretability trade-offs>`
- **Where it fails:** `<segments and error modes>`
- **Confidence:** `<is the gap to the runner-up larger than the noise?>`
- **Reproduced by:** commit `<hash>` + `dvc checkout`
- **Next steps:** `<what would improve it most>`